# Week 87, Day 6: Review and plan

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w087d6.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
import statistics
velocity = [8, 13, 9, 11]
print(statistics.mean(velocity[-3:]), min(velocity[-3:]), max(velocity[-3:]))

**Predict before you run:** what will this print?



In [ ]:
import math
backlog_points = 40
for v in (9, 11, 13):
    print(v, math.ceil(backlog_points / v))

**Predict before you run:** what will this print?



In [ ]:
done = {"S1"}
backlog = [("S3", {"S1"}), ("S4", {"S1", "S6"}), ("S6", set())]
print([s for s, needs in backlog if needs <= done])

**Predict before you run:** what will this print?



In [ ]:
total, days = 20, 10
print([round(total - total * d / days, 1) for d in range(0, days + 1, 2)])

**Predict before you run:** what will this print?



In [ ]:
notes = ["went well: CI from day one", "didn't: spec took twice as long", "try: timebox writing to 2 hours",
         "didn't: no tests for storage"]
print({kind: sum(n.startswith(kind) for n in notes) for kind in ("went well", "didn't", "try")})

### Exercise: Is the slice really done?

Write `slice_report(story_ids, scenarios, tests)`: `scenarios` maps story ids to their number of scenarios and `tests` is a list of test names (`test_s1_...`). Return a dict from each story id (in order) to `"done"` if it has at least one scenario and at least as many tests as scenarios, `"needs tests"` if it has scenarios but fewer tests, and `"needs scenarios"` if it has none.

In [ ]:
def slice_report(story_ids, scenarios, tests):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_slice():\n    \"scenarios and tests together\"\n    tests = [\"test_s1_shortest_route\", \"test_s1_no_route\", \"test_s1_unknown_place\", \"test_s2_time_with_climbing\", \"test_helpers\"]\n    assert slice_report([\"S1\", \"S2\", \"S3\", \"S4\"], {\"S1\": 2, \"S2\": 2, \"S3\": 1}, tests) == {\n        \"S1\": \"done\", \"S2\": \"needs tests\", \"S3\": \"needs tests\", \"S4\": \"needs scenarios\"}")

### Exercise: Which layer?

Write `layer_of(responsibility)` for Waymark, returning `"domain"`, `"storage"`, `"service"` or `"interface"` from keywords in the (lower-cased) text, checking in this order: anything mentioning `sql`, `table`, `database` or `rows` is storage; `http`, `status code`, `argparse`, `exit code`, `json` or `command line` is interface; `transaction`, `use case` or `coordinate` is service; everything else is domain. Then `misplaced(assignments)`: for a dict from responsibility to the layer someone put it in, the sorted responsibilities whose layer differs from `layer_of`.

In [ ]:
def layer_of(responsibility):
    ___

def misplaced(assignments):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_layers():\n    \"one job per layer\"\n    assert layer_of(\"Naismith's rule\") == \"domain\" and layer_of(\"Load paths from the path table\") == \"storage\"\n    assert layer_of(\"Turn NoRoute into a 404 status code\") == \"interface\" and layer_of(\"Save a walk in one transaction\") == \"service\"\n    plan = {\"Naismith's rule\": \"domain\", \"Parse argparse options\": \"service\", \"SQL for saved routes\": \"storage\",\n            \"Shortest route search\": \"domain\", \"Return JSON errors\": \"domain\"}\n    assert misplaced(plan) == [\"Parse argparse options\", \"Return JSON errors\"]")

### Exercise: Plan the sprint

Write `velocity(history, window=3)`: from a list of points finished per past sprint, `(average, low, high)` over the last `window` sprints (the average rounded to 1 place); with no history, `(0.0, 0, 0)`. Then `plan_sprint(backlog, done, capacity)`: `backlog` is a list of dicts with `id`, `points`, `priority` (a number: lower is more important) and `needs` (a list of ids). Go through the backlog by priority (ties in the given order) and include a story if its points fit in what's left of `capacity` and every story it needs is in `done` or already included; otherwise it's **blocked** (if a need isn't met) or **deferred** (if it doesn't fit). Return a dict with `included` (ids in the order chosen), `points` (their total), `blocked`, `deferred` (ids in backlog priority order) and `goal`: the titles of the included stories joined as `"S3, S4 and S6"` (or `"nothing"`). Finally `forecast(backlog_points, history, window=3)`: the number of sprints to finish `backlog_points` at the average, the high and the low velocity, as a dict with `expected`, `best` and `worst` (each rounded **up**); raise `ValueError("no velocity yet")` if there's no history.

In [ ]:
import math

def velocity(history, window=3):
    ___

def plan_sprint(backlog, done, capacity):
    ___

def forecast(backlog_points, history, window=3):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("BACKLOG = [\n    {\"id\": \"S3\", \"points\": 3, \"priority\": 1, \"needs\": [\"S1\"]},\n    {\"id\": \"S5\", \"points\": 5, \"priority\": 2, \"needs\": [\"S3\"]},\n    {\"id\": \"S4\", \"points\": 8, \"priority\": 2, \"needs\": [\"S1\", \"S7\"]},\n    {\"id\": \"S6\", \"points\": 2, \"priority\": 3, \"needs\": []},\n    {\"id\": \"S7\", \"points\": 5, \"priority\": 4, \"needs\": []},\n    {\"id\": \"S8\", \"points\": 1, \"priority\": 5, \"needs\": [\"S6\"]},\n]\n\ndef test_velocity():\n    \"the last three sprints\"\n    assert velocity([8, 13, 9, 11]) == (11.0, 9, 13)\n    assert velocity([5]) == (5.0, 5, 5) and velocity([]) == (0.0, 0, 0)\n    assert velocity([10, 4, 7, 9], window=2) == (8.0, 7, 9)\n\ndef test_plan():\n    \"priority, capacity and dependencies\"\n    p = plan_sprint(BACKLOG, {\"S1\", \"S2\"}, 11)\n    assert p == {\"included\": [\"S3\", \"S5\", \"S6\", \"S8\"], \"points\": 11, \"blocked\": [\"S4\"], \"deferred\": [\"S7\"],\n                 \"goal\": \"S3, S5, S6 and S8\"}\n    p = plan_sprint(BACKLOG, {\"S1\", \"S2\"}, 20)\n    assert p[\"included\"] == [\"S3\", \"S5\", \"S6\", \"S7\", \"S8\"] and p[\"blocked\"] == [\"S4\"], \"S7 comes after S4, so S4 is still blocked\"\n    p = plan_sprint(BACKLOG, {\"S2\"}, 30)\n    assert p[\"included\"] == [\"S6\", \"S7\", \"S8\"] and p[\"blocked\"] == [\"S3\", \"S5\", \"S4\"]\n    assert plan_sprint(BACKLOG, {\"S1\"}, 0)[\"goal\"] == \"nothing\"\n    assert plan_sprint([{\"id\": \"S9\", \"points\": 2, \"priority\": 1, \"needs\": []}], set(), 5)[\"goal\"] == \"S9\"\n\ndef test_forecast():\n    \"a range, not a promise\"\n    assert forecast(40, [8, 13, 9, 11]) == {\"expected\": 4, \"best\": 4, \"worst\": 5}\n    assert forecast(0, [5]) == {\"expected\": 0, \"best\": 0, \"worst\": 0}\n    try:\n        forecast(10, [])\n    except ValueError as e:\n        assert str(e) == \"no velocity yet\"\n    else:\n        raise AssertionError(\"no history\")")

> **What the plan leaves out.** With a capacity of 11 points, S4 (round walks) is blocked: it needs S7 (a faster path search), which is low priority. The planner did exactly what it was told, and the result is a sprint that skips the riskiest story. That's a sign the **priorities** are wrong, not the planner: if S4 matters, S7 must move up. A planning tool makes the consequences of your priorities visible; it can't choose them for you. Which of your capstone's stories is blocked by something you've ranked too low?

### Exercise: Bonus: a burndown

Write a class `Burndown(total, days)`. `record(day, remaining)` stores the points remaining at the end of `day` (from 1 to `days`; `ValueError("no such day")` otherwise; recording a day again replaces it). `ideal(day)` is `total − total × day / days`, rounded to 1 place. `status()` uses the latest recorded day: `"ahead"` if remaining is below the ideal by more than 10% of `total`, `"behind"` if above it by more than 10% of `total`, otherwise `"on track"`; `"not started"` with nothing recorded. `stalled(n=2)` is `True` if the last `n + 1` recorded days (consecutive or not) all have the same remaining points. `chart()` returns a matplotlib figure with one axes, titled `"Sprint burndown"`, with two lines: the ideal from day 0 to `days` (labelled `"ideal"`, dashed) and the actual from day 0 (at `total`) through each recorded day in order (labelled `"actual"`, with markers), a legend, and axis labels `"day"` and `"points remaining"`, with the y axis starting at 0.

In [ ]:
import matplotlib.pyplot as plt

class Burndown:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_burndown():\n    \"ideal, status, stalls\"\n    b = Burndown(20, 5)\n    assert [b.ideal(d) for d in range(6)] == [20.0, 16.0, 12.0, 8.0, 4.0, 0.0]\n    assert b.status() == \"not started\"\n    b.record(1, 17)\n    assert b.status() == \"on track\"\n    b.record(2, 17)\n    b.record(3, 17)\n    assert b.status() == \"behind\" and b.stalled()\n    b.record(3, 9)\n    assert b.status() == \"on track\" and not b.stalled()\n    b.record(4, 1)\n    assert b.status() == \"ahead\"\n    try:\n        b.record(6, 0)\n    except ValueError as e:\n        assert str(e) == \"no such day\"\n    else:\n        raise AssertionError(\"day 6\")\n\ndef test_chart():\n    \"two honest lines\"\n    b = Burndown(20, 5)\n    for day, left in [(1, 18), (2, 15), (3, 15)]:\n        b.record(day, left)\n    fig = b.chart()\n    ax = fig.axes[0]\n    ideal, actual = ax.lines\n    assert list(ideal.get_xdata()) == [0, 5] and list(ideal.get_ydata()) == [20, 0] and ideal.get_linestyle() == \"--\"\n    assert list(actual.get_xdata()) == [0, 1, 2, 3] and list(actual.get_ydata()) == [20, 18, 15, 15]\n    assert [t.get_text() for t in ax.get_legend().get_texts()] == [\"ideal\", \"actual\"]\n    assert ax.get_title() == \"Sprint burndown\" and ax.get_xlabel() == \"day\" and ax.get_ylim()[0] == 0")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

## Warm-up

## Mixed review

## Looking back

A **retrospective** turns a week's experience into changes. Keep it short and specific, and do it even alone:

1. **What went well?** (Keep doing it.) "CI from the first commit caught a missing `__init__.py`."
2. **What didn't?** (Facts, not blame, including blaming yourself.) "Writing the spec took eight hours instead of four."
3. **What will I try next week?** One to three **actions**, each specific, owned and checkable: "Timebox spec changes to two hours and list open questions instead of solving them all." Not "be more efficient".

Look at the numbers too: estimated against actual hours for each piece of work (your estimates will improve only if you compare), how many tests and how many CI failures, and anything you deferred. Then check the week's work against a **definition of done**, the list every story must meet before it counts. Waymark's:

- the acceptance scenarios pass as automated tests named after the story;
- the code is reviewed (your own pull request, read the next day) and merged with green CI;
- the README and spec are updated if behaviour changed;
- no new warnings from the linter, and the architecture check passes.

## Planning the sprint

The next three weeks are **build sprints**: one week each, with a goal, a plan, daily progress, a review and a retrospective.

- **The backlog** is every story not yet done, in priority order (from Day 1's MoSCoW, revised by what you've learned).
- **Size** each story in **points**: a relative measure of effort and uncertainty (1, 2, 3, 5, 8, 13), not hours. Relative sizes are easier to judge consistently ("S3 is about twice S5") and absorb your personal speed.
- **Velocity** is how many points you actually finish in a sprint. With no history, guess cautiously; from now on, plan with the **average of your last three sprints**, and use the lowest and highest of them for a pessimistic and optimistic forecast.
- **Dependencies**: some stories need others first (Waymark's S5, logging walks, needs S3's saved routes). A story can go into the sprint only if what it depends on is done or earlier in the same sprint.
- **A sprint goal** in one sentence ("walkers can save routes and plan round walks") tells you what matters when something has to give.

Then fill the sprint in priority order, skipping anything too big for what's left or blocked by a dependency, and stop at the capacity. Leave the rest: an honest plan you finish beats an ambitious one you don't.

## Bonus: a burndown

A **burndown** chart shows the points remaining in the sprint at the end of each day against an **ideal line** falling evenly from the total to zero. Above the line, you're behind; a flat stretch means nothing is finishing (often a story that's too big, or stuck); a line that jumps **up** means work was added mid-sprint. It turns a feeling ("I think it's going fine") into a fact you can act on by Wednesday instead of Sunday.

## Weekly quiz

## Week 87 review

Tick off each one honestly. If one feels shaky, go back to that day.

- I can judge capstone ideas with a weighted matrix, estimate with three-point estimates, cut to an MVP with MoSCoW, rank risks and write a pitch. ([Day 1](w087d1.html))
- I can write a specification with users, stories, Given/When/Then scenarios, measurable requirements, an out-of-scope list and a glossary, and check it. ([Day 2](w087d2.html))
- I can divide a program into layers, check the dependency rule and cycles from its imports, design a schema, map routes to stories and record decisions. ([Day 3](w087d3.html))
- I can set up a repository with CI, ignore rules and conventional commits, and build a walking skeleton. ([Day 4](w087d4.html))
- I can build a vertical slice through every layer, tested at each level, and spike the riskiest part. ([Day 5](w087d5.html))
- I can run a retrospective, measure velocity, plan a sprint around dependencies and track it with a burndown. (Today)

## Your capstone today

On your own computer, in `docs/sprints/`:

1. **Retrospective** (`week-1-retro.md`): what went well, what didn't, and one to three specific actions. Include estimated against actual hours for each piece of the week's work.
2. **Definition of done**: write yours in the README's development section (Waymark's is a good start).
3. **The backlog**: every remaining story, sized in points, with priorities and dependencies. Check the ranking: is anything important blocked by something ranked low?
4. **Sprint 1 plan** (`sprint-1.md`): a goal in one sentence, the stories from `plan_sprint` with a cautious capacity (you have no velocity yet: take about two thirds of what your hours suggest), and a burndown you'll update each day you work.

Tomorrow is a **rest day**. Next week is **Capstone 2: the first build sprint**: three days of building, a day of tests, a code review and a sprint review.